# Dynamically Weighted Stage-B Ensemble — Two-Stage Hierarchical NIDS

**Extends Stage 6 (Model Improvement)**, building on `two_stage_stageB_weighting.ipynb`
(current best: Stage-B tempered weights α=0.25, test macro-F1 **0.6157**).

## The idea

A static ensemble asks *which model is best overall*. A **dynamically weighted** ensemble asks
*which model is best right here* — competence is estimated per test instance from the model's
accuracy in the local neighbourhood of that instance, and the base models' probabilities are
combined with those per-instance weights:

$$P(k \mid x) \;=\; \frac{\sum_m w_m(x)\, P_m(k \mid x)}{\sum_m w_m(x)},
\qquad w_m(x) = \text{accuracy of } m \text{ on the } K \text{ neighbours of } x$$

The motivating observation is that RF and XGBoost had **inverted** precision/recall profiles on this
data. But that was measured under `class_weight='balanced'`, which the weighting study showed is a
bad operating point — so **the premise has to be re-verified, not assumed**. Section 5 does that,
and computes an **oracle bound** that caps what any per-instance scheme can achieve. If the oracle
is close to the best single model, no dynamic combiner can help and this notebook reports that
instead of chasing it.

## Scope and protocol

- Dynamic weighting is applied to **Stage B only**. The gate is at 0.9966 recall — it is not the
  bottleneck and is left untouched.
- All base learners use the **α=0.25** weighting established as best in the weighting study.
- Competence estimation needs data the base models never saw, so `val` is halved:
  **DSEL** (competence source) and **SEL** (variant/K selection). The test set is touched once.
- Distances are computed in **RobustScaler** space — raw CICFlowMeter features are heavy-tailed and a
  kNN in unscaled space is decided entirely by the largest-magnitude columns.

> **Fairness note.** Base models here train on `train` only (195,950 rows), because `val` is needed
> as DSEL. The 0.6157 reference was refit on train+val (244,938). Section 8 therefore reports a
> train-only XGBoost as the apples-to-apples baseline alongside it.

## 1. Setup and splits

Identical data, prune and seeds as the weighting study, so `train`/`val`/`test` match row-for-row.
`val` is then halved into DSEL and SEL.

In [1]:
%pip install xgboost -q

import numpy as np, pandas as pd, time, warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import RobustScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix)
from xgboost import XGBClassifier

LABELS = {0:"Benign", 1:"Analysis", 2:"Backdoor", 3:"DoS", 4:"Exploits",
          5:"Fuzzers", 6:"Generic", 7:"Reconnaissance", 8:"Shellcode", 9:"Worms"}
RS = 42
ALPHA = 0.25          # winner of the weighting study

X = pd.read_csv("Resource/CIC-UNSW-Data.csv")
y = pd.read_csv("Resource/CIC-UNSW-Label.csv")["Label"]
df = pd.concat([X, y], axis=1).drop_duplicates().reset_index(drop=True)
X, y = df[X.columns], df["Label"]

nun   = X.nunique(); zero = nun[nun <= 1].index.tolist()
corr  = X.drop(columns=zero).corr()
upper = corr.abs().where(np.triu(np.ones(corr.shape, bool), k=1))
red   = [c for c in upper.columns if (upper[c] >= 0.95).any()]
X = X.drop(columns=list(dict.fromkeys(zero + red)))

X_trf, X_test, y_trf, y_test = train_test_split(X, y, test_size=0.2, random_state=RS, stratify=y)
X_tr,  X_val,  y_tr,  y_val  = train_test_split(X_trf, y_trf, test_size=0.2, random_state=RS, stratify=y_trf)
# halve val: DSEL estimates competence, SEL selects the variant and K
X_dsel, X_sel, y_dsel, y_sel = train_test_split(X_val, y_val, test_size=0.5, random_state=RS, stratify=y_val)

print("features:", X.shape[1])
for n, s in [("train", y_tr), ("DSEL", y_dsel), ("SEL", y_sel), ("test", y_test)]:
    print("%6s: %7d rows  (%d attack)" % (n, len(s), (s != 0).sum()))

Note: you may need to restart the kernel to use updated packages.
features: 44
 train:  195950 rows  (51949 attack)
  DSEL:   24494 rows  (6494 attack)
   SEL:   24494 rows  (6493 attack)
  test:   61235 rows  (16234 attack)


## 2. Fixed gate and Stage-B data

The gate is trained on `train` and shared by every variant, so all differences are Stage B's.

In [2]:
def make_xgb(**kw):
    p = dict(n_estimators=400, max_depth=8, learning_rate=0.1, subsample=0.9,
             colsample_bytree=0.9, tree_method="hist", eval_metric="mlogloss",
             n_jobs=-1, random_state=RS)
    p.update(kw)
    return XGBClassifier(**p)

gate = make_xgb()
gate.fit(X_tr, (y_tr != 0).astype(int),
         sample_weight=compute_sample_weight("balanced", (y_tr != 0).astype(int)))
gate_sel, gate_test = gate.predict(X_sel), gate.predict(X_test)
print("gate  SEL attack recall %.4f / precision %.4f"
      % (recall_score((y_sel != 0).astype(int), gate_sel),
         precision_score((y_sel != 0).astype(int), gate_sel)))

attack_labels = sorted(l for l in y_tr.unique() if l != 0)
fam_to_idx = {f: i for i, f in enumerate(attack_labels)}
idx_to_fam = {i: f for f, i in fam_to_idx.items()}
NC  = len(attack_labels)
FAM = [LABELS[idx_to_fam[i]] for i in range(NC)]

def attack_only(Xa, ya):
    m = ya != 0
    return Xa[m].reset_index(drop=True), ya[m].map(fam_to_idx).reset_index(drop=True)

X_tr_att,   y_tr_att   = attack_only(X_tr,   y_tr)
X_dsel_att, y_dsel_att = attack_only(X_dsel, y_dsel)
print("Stage-B train rows:", len(y_tr_att), "| DSEL attack rows:", len(y_dsel_att))

gate  SEL attack recall 0.9961 / precision 0.9232
Stage-B train rows: 51949 | DSEL attack rows: 6494


## 3. Base learners

Three different model families for diversity — a dynamic combiner can only exploit disagreement, and
disagreement comes from different inductive biases, not from reseeding one algorithm. All three get
the same **α=0.25** class weights.

In [3]:
def sample_w(y_arr, wc):
    w = wc[y_arr].astype(float)
    return w * len(w) / w.sum()

def wc_alpha(n, a):
    w = (n.sum() / n) ** a
    return w / w.mean()

n_tr = np.bincount(y_tr_att.values, minlength=NC)
WC   = wc_alpha(n_tr, ALPHA)
sw   = sample_w(y_tr_att.values, WC)

BASE = {
    "XGB": make_xgb(),
    "RF":  RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RS),
    "ET":  ExtraTreesClassifier(n_estimators=300, n_jobs=-1, random_state=RS),
}
for name, m in BASE.items():
    t = time.time(); m.fit(X_tr_att, y_tr_att, sample_weight=sw)
    print("%-4s fitted in %5.0fs" % (name, time.time() - t))
MODELS = list(BASE)
M = len(MODELS)

XGB  fitted in    10s
RF   fitted in     4s
ET   fitted in     2s


## 4. Precompute probabilities and neighbourhoods

Everything expensive happens once here; every variant below is then a cheap re-weighting.

Stage-B probabilities are computed for **all** rows of a query set (not just true attacks), because
end-to-end routing needs a family prediction for every row the gate flags.

In [4]:
KMAX = 50
scaler = RobustScaler().fit(X_tr_att)              # heavy-tailed features -> scale before any kNN
Z_dsel = scaler.transform(X_dsel_att)
nn = NearestNeighbors(n_neighbors=KMAX, algorithm="brute", n_jobs=-1).fit(Z_dsel)

# per-model correctness on DSEL -- the raw material for every competence estimate
dsel_pred = {k: m.predict(X_dsel_att) for k, m in BASE.items()}
CORRECT   = [dsel_pred[k] == y_dsel_att.values for k in MODELS]
y_dsel_arr = y_dsel_att.values
for k in MODELS:
    print("%-4s DSEL accuracy %.4f" % (k, (dsel_pred[k] == y_dsel_arr).mean()))

def prep(Xq):
    """probabilities from every base model + neighbourhood of each query row in DSEL."""
    t = time.time()
    probas = [BASE[k].predict_proba(Xq) for k in MODELS]
    dist, idx = nn.kneighbors(scaler.transform(Xq))
    print("prepped %d rows in %.0fs" % (len(Xq), time.time() - t))
    return dict(probas=probas, dist=dist, idx=idx)

Q_SEL  = prep(X_sel)
Q_TEST = prep(X_test)

XGB  DSEL accuracy 0.7381
RF   DSEL accuracy 0.7196
ET   DSEL accuracy 0.7114
prepped 24494 rows in 1s
prepped 61235 rows in 1s


## 5. Is there anything to exploit? — complementarity and the oracle bound

The decisive diagnostic, and the reason to run it *before* building combiners:

- **Disagreement rate** — if the models agree almost everywhere, per-instance weighting has no
  material to work with.
- **Oracle macro-F1** — score obtained if an omniscient selector picked a correct model whenever
  *any* base model is right. This is a hard **upper bound on every per-instance selection scheme**,
  dynamic weighting included. If it sits near the best single model, the answer is already no.

In [5]:
sel_att   = (y_sel != 0).values
y_sel_fam = y_sel[y_sel != 0].map(fam_to_idx).values
preds_sel = [p[sel_att].argmax(1) for p in Q_SEL["probas"]]

print("--- single models on SEL attack rows (Stage-B macro-F1) ---")
for name, pr in zip(MODELS, preds_sel):
    print("  %-4s  mF1=%.4f  mP=%.4f  mR=%.4f"
          % (name, f1_score(y_sel_fam, pr, average="macro", zero_division=0),
             precision_score(y_sel_fam, pr, average="macro", zero_division=0),
             recall_score(y_sel_fam, pr, average="macro", zero_division=0)))

print("\n--- pairwise disagreement / who is right when they differ ---")
for i in range(M):
    for j in range(i + 1, M):
        d = preds_sel[i] != preds_sel[j]
        ci = (preds_sel[i][d] == y_sel_fam[d]).mean()
        cj = (preds_sel[j][d] == y_sel_fam[d]).mean()
        print("  %-4s vs %-4s : disagree on %5.2f%% | %s right %5.2f%%, %s right %5.2f%%, both wrong %5.2f%%"
              % (MODELS[i], MODELS[j], 100*d.mean(), MODELS[i], 100*ci,
                 MODELS[j], 100*cj, 100*(1-ci-cj)))

any_right = np.zeros(len(y_sel_fam), bool)
for pr in preds_sel:
    any_right |= (pr == y_sel_fam)
oracle = np.where(any_right, y_sel_fam, preds_sel[0])
best_single = max(f1_score(y_sel_fam, pr, average="macro", zero_division=0) for pr in preds_sel)
print("\noracle Stage-B macro-F1 : %.4f" % f1_score(y_sel_fam, oracle, average="macro", zero_division=0))
print("best single model       : %.4f" % best_single)
print("headroom for ANY per-instance selector: %.4f"
      % (f1_score(y_sel_fam, oracle, average="macro", zero_division=0) - best_single))
print("at least one model correct on %.2f%% of attack rows" % (100*any_right.mean()))

--- single models on SEL attack rows (Stage-B macro-F1) ---
  XGB   mF1=0.5995  mP=0.6641  mR=0.5665
  RF    mF1=0.5782  mP=0.6740  mR=0.5273
  ET    mF1=0.5665  mP=0.6412  mR=0.5232

--- pairwise disagreement / who is right when they differ ---
  XGB  vs RF   : disagree on 13.20% | XGB right 44.81%, RF right 29.64%, both wrong 25.55%
  XGB  vs ET   : disagree on 16.42% | XGB right 45.31%, ET right 28.71%, both wrong 25.98%
  RF   vs ET   : disagree on  7.41% | RF right 41.37%, ET right 31.60%, both wrong 27.03%

oracle Stage-B macro-F1 : 0.6799
best single model       : 0.5995
headroom for ANY per-instance selector: 0.0804
at least one model correct on 79.09% of attack rows


## 6. Combiners

`comp_ola` — **Overall Local Accuracy**: each model's accuracy over the K nearest DSEL neighbours.
Optional refinements: neighbours weighted by inverse distance, and DSEL rows weighted by the same
α=0.25 class weights so that competence on *rare* families counts more than competence on Exploits
(plain local accuracy is otherwise dominated by the big families — the exact failure mode macro-F1
exists to punish).

`comp_lca` — **Local Class Accuracy**: accuracy restricted to neighbours whose true family equals
what the model is predicting for this query. Falls back to OLA when no such neighbour exists.

`p` sharpens the weights ($w^p$): $p=0$ is a plain average, large $p$ approaches hard selection of
the locally-best model.

In [6]:
def comp_ola(Q, k, class_weighted=False, dist_weighted=False):
    nb = Q["idx"][:, :k]
    wn = WC[y_dsel_arr[nb]] if class_weighted else np.ones(nb.shape)
    if dist_weighted:
        wn = wn / (Q["dist"][:, :k] + 1e-6)
    denom = wn.sum(1)
    W = np.empty((nb.shape[0], M))
    for m in range(M):
        W[:, m] = (CORRECT[m][nb] * wn).sum(1) / denom
    return W

def comp_lca(Q, k):
    nb = Q["idx"][:, :k]
    yn = y_dsel_arr[nb]
    W  = np.empty((nb.shape[0], M))
    for m in range(M):
        pq   = Q["probas"][m].argmax(1)
        mask = yn == pq[:, None]
        cnt  = mask.sum(1)
        W[:, m] = np.where(cnt > 0,
                           (CORRECT[m][nb] & mask).sum(1) / np.maximum(cnt, 1),
                           CORRECT[m][nb].mean(1))
    return W

def fuse(Q, W, p=1.0):
    """per-instance weighted soft vote -> family index per row."""
    Wp = np.power(np.clip(W, 1e-6, None), p)
    Wp = Wp / Wp.sum(1, keepdims=True)
    P = np.zeros_like(Q["probas"][0])
    for m in range(M):
        P += Wp[:, m:m + 1] * Q["probas"][m]
    return P.argmax(1)

def score(fam_pred_all, ye, gate_pred, tag):
    """fam_pred_all = Stage-B family index for EVERY row of the query set."""
    att    = (ye != 0).values
    ye_fam = ye[ye != 0].map(fam_to_idx).values
    yp = np.zeros(len(ye), int)
    g  = gate_pred == 1
    yp[g] = np.vectorize(idx_to_fam.get)(fam_pred_all[g])
    return dict(variant=tag,
                stageB_mF1=round(f1_score(ye_fam, fam_pred_all[att], average="macro", zero_division=0), 4),
                e2e_mF1=round(f1_score(ye, yp, average="macro", zero_division=0), 4),
                e2e_mP=round(precision_score(ye, yp, average="macro", zero_division=0), 4),
                e2e_mR=round(recall_score(ye, yp, average="macro", zero_division=0), 4),
                e2e_acc=round(float((yp == ye.values).mean()), 4))

### 6.1 Variant sweep on SEL

Single models and static fusion are included as the honest controls — a dynamic scheme that cannot
beat a plain average of the same three models has bought nothing but complexity.

In [7]:
def run_all(Q, ye, gate_pred):
    out, eq = [], np.ones((len(ye), M))
    for m, name in enumerate(MODELS):                       # controls: single models
        out.append(score(Q["probas"][m].argmax(1), ye, gate_pred, "single: " + name))
    out.append(score(fuse(Q, eq), ye, gate_pred, "static: equal average"))
    for k in (5, 10, 20, 50):                               # dynamic: OLA
        out.append(score(fuse(Q, comp_ola(Q, k)), ye, gate_pred, f"dyn OLA k={k}"))
    for k in (10, 30):                                      # dynamic: class-weighted OLA
        out.append(score(fuse(Q, comp_ola(Q, k, class_weighted=True)), ye, gate_pred,
                         f"dyn OLA-cw k={k}"))
    out.append(score(fuse(Q, comp_ola(Q, 30, class_weighted=True, dist_weighted=True)), ye,
                     gate_pred, "dyn OLA-cw+dist k=30"))
    for k in (10, 30):                                      # dynamic: LCA
        out.append(score(fuse(Q, comp_lca(Q, k)), ye, gate_pred, f"dyn LCA k={k}"))
    for p in (3, 8):                                        # sharpened -> towards hard selection
        out.append(score(fuse(Q, comp_ola(Q, 30, class_weighted=True), p=p), ye, gate_pred,
                         f"dyn OLA-cw k=30 p={p}"))
    return out

sel_res = run_all(Q_SEL, y_sel, gate_sel)
sel_df  = (pd.DataFrame(sel_res).sort_values("e2e_mF1", ascending=False).reset_index(drop=True))
print(sel_df.to_string(index=False))

              variant  stageB_mF1  e2e_mF1  e2e_mP  e2e_mR  e2e_acc
  dyn OLA-cw k=30 p=8      0.6026   0.6212  0.6718  0.6036   0.9063
         dyn LCA k=30      0.5986   0.6203  0.7013  0.5887   0.9078
         dyn LCA k=10      0.5979   0.6197  0.6870  0.5900   0.9077
          single: XGB      0.5995   0.6167  0.6589  0.6062   0.9078
  dyn OLA-cw k=30 p=3      0.5974   0.6165  0.6662  0.5970   0.9054
         dyn OLA k=10      0.5945   0.6137  0.6654  0.5926   0.9046
          dyn OLA k=5      0.5960   0.6135  0.6618  0.5948   0.9047
      dyn OLA-cw k=10      0.5952   0.6129  0.6621  0.5938   0.9046
         dyn OLA k=50      0.5918   0.6109  0.6623  0.5895   0.9045
 dyn OLA-cw+dist k=30      0.5912   0.6101  0.6593  0.5899   0.9047
         dyn OLA k=20      0.5904   0.6098  0.6586  0.5893   0.9047
      dyn OLA-cw k=30      0.5906   0.6094  0.6570  0.5897   0.9045
static: equal average      0.5897   0.6090  0.6596  0.5878   0.9041
           single: RF      0.5782   0.5984  0.66

## 7. Pick the winner on SEL

Selection is on `e2e_mF1`, and it is only reported as a *dynamic-ensemble* win if it beats both the
best single model and the static average — otherwise the honest conclusion is that the extra
machinery is not earning its cost.

In [8]:
BESTV   = sel_df.iloc[0]["variant"]
best_dyn    = sel_df[sel_df.variant.str.startswith("dyn")].iloc[0]
best_single_row = sel_df[sel_df.variant.str.startswith("single")].iloc[0]
static_row  = sel_df[sel_df.variant == "static: equal average"].iloc[0]

print("best overall       : %-24s %.4f" % (BESTV, sel_df.iloc[0]["e2e_mF1"]))
print("best dynamic       : %-24s %.4f" % (best_dyn["variant"], best_dyn["e2e_mF1"]))
print("best single model  : %-24s %.4f" % (best_single_row["variant"], best_single_row["e2e_mF1"]))
print("static average     : %-24s %.4f" % ("equal", static_row["e2e_mF1"]))
print("\ndynamic vs best single : %+.4f" % (best_dyn["e2e_mF1"] - best_single_row["e2e_mF1"]))
print("dynamic vs static      : %+.4f" % (best_dyn["e2e_mF1"] - static_row["e2e_mF1"]))

best overall       : dyn OLA-cw k=30 p=8      0.6212
best dynamic       : dyn OLA-cw k=30 p=8      0.6212
best single model  : single: XGB              0.6167
static average     : equal                    0.6090

dynamic vs best single : +0.0045
dynamic vs static      : +0.0122


### 7.1 Where do the per-instance weights actually go?

A dynamic ensemble that assigns near-identical weights everywhere is a static ensemble wearing a
costume. This shows the mean weight per model and how much it varies across instances — real
dispersion is what makes the approach worth its runtime.

In [9]:
W = comp_ola(Q_SEL, 30, class_weighted=True)
Wn = W / W.sum(1, keepdims=True)
print(pd.DataFrame({"mean weight": Wn.mean(0), "std across instances": Wn.std(0),
                    "min": Wn.min(0), "max": Wn.max(0),
                    "% of rows this model leads": [100*(Wn.argmax(1) == m).mean() for m in range(M)]},
                   index=MODELS).round(4).to_string())

     mean weight  std across instances     min     max  % of rows this model leads
XGB       0.3348                0.0235  0.2194  0.5565                     43.3780
RF        0.3321                0.0129  0.1871  0.4123                     12.5827
ET        0.3331                0.0163  0.2024  0.4177                     44.0394


## 8. Final evaluation — test set, once

The SEL-selected variant is applied to test with DSEL unchanged. Base models stay train-only, so the
train-only single XGBoost is the like-for-like reference; the 0.6157 from the weighting study
(refit on train+val) is shown for context.

In [10]:
VARIANT_FN = {
    "static: equal average":      lambda Q: fuse(Q, np.ones((len(Q["dist"]), M))),
    "dyn OLA-cw+dist k=30":       lambda Q: fuse(Q, comp_ola(Q, 30, class_weighted=True, dist_weighted=True)),
}
for k in (5, 10, 20, 50):
    VARIANT_FN[f"dyn OLA k={k}"]    = (lambda kk: lambda Q: fuse(Q, comp_ola(Q, kk)))(k)
for k in (10, 30):
    VARIANT_FN[f"dyn OLA-cw k={k}"] = (lambda kk: lambda Q: fuse(Q, comp_ola(Q, kk, class_weighted=True)))(k)
    VARIANT_FN[f"dyn LCA k={k}"]    = (lambda kk: lambda Q: fuse(Q, comp_lca(Q, kk)))(k)
for p in (3, 8):
    VARIANT_FN[f"dyn OLA-cw k=30 p={p}"] = (lambda pp: lambda Q: fuse(Q, comp_ola(Q, 30, class_weighted=True), p=pp))(p)
for m, name in enumerate(MODELS):
    VARIANT_FN["single: " + name] = (lambda mm: lambda Q: Q["probas"][mm].argmax(1))(m)

fam_test = VARIANT_FN[BESTV](Q_TEST)
final    = score(fam_test, y_test, gate_test, BESTV)
xgb_only = score(Q_TEST["probas"][MODELS.index("XGB")].argmax(1), y_test, gate_test,
                 "single: XGB (train-only)")

print("--- TEST ---")
print(pd.DataFrame([final, xgb_only]).to_string(index=False))
print("\ndynamic ensemble vs train-only XGBoost : %+.4f macro-F1"
      % (final["e2e_mF1"] - xgb_only["e2e_mF1"]))
print("vs weighting-study best (0.6157, train+val refit): %+.4f" % (final["e2e_mF1"] - 0.6157))

--- TEST ---
                 variant  stageB_mF1  e2e_mF1  e2e_mP  e2e_mR  e2e_acc
     dyn OLA-cw k=30 p=8      0.5975   0.6118  0.6609  0.5932   0.9049
single: XGB (train-only)      0.5966   0.6125  0.6580  0.6002   0.9068

dynamic ensemble vs train-only XGBoost : -0.0007 macro-F1
vs weighting-study best (0.6157, train+val refit): -0.0039


### 8.1 Per-class report and confusion matrix (test)

In [11]:
yp = np.zeros(len(y_test), int)
g  = gate_test == 1
yp[g] = np.vectorize(idx_to_fam.get)(fam_test[g])
print(classification_report(y_test, yp, labels=list(range(10)),
                            target_names=[LABELS[i] for i in range(10)], digits=3, zero_division=0))
print(pd.DataFrame(confusion_matrix(y_test, yp, labels=list(range(10))),
                   index=[LABELS[i] for i in range(10)],
                   columns=[LABELS[i] for i in range(10)]).to_string())

                precision    recall  f1-score   support

        Benign      0.999     0.968     0.983     45001
      Analysis      0.416     0.545     0.472        77
      Backdoor      0.800     0.533     0.640        90
           DoS      0.642     0.294     0.404       883
      Exploits      0.765     0.779     0.772      6182
       Fuzzers      0.597     0.853     0.702      5279
       Generic      0.842     0.762     0.800       899
Reconnaissance      0.657     0.581     0.617      2406
     Shellcode      0.393     0.262     0.314       370
         Worms      0.500     0.354     0.415        48

      accuracy                          0.905     61235
     macro avg      0.661     0.593     0.612     61235
  weighted avg      0.914     0.905     0.906     61235

                Benign  Analysis  Backdoor  DoS  Exploits  Fuzzers  Generic  Reconnaissance  Shellcode  Worms
Benign           43547        27         4   10       193     1047       17             137         16 

### 8.2 Full test table (all variants, for the ablation appendix)

Selection already happened on SEL; this is reported for completeness and **must not** be used to
pick a different winner after the fact.

In [12]:
test_all = pd.DataFrame(run_all(Q_TEST, y_test, gate_test)).sort_values("e2e_mF1", ascending=False)
print(test_all.to_string(index=False))
print("\nSEL-selected variant was: %s (rank %d on test)"
      % (BESTV, 1 + list(test_all.variant).index(BESTV)))

              variant  stageB_mF1  e2e_mF1  e2e_mP  e2e_mR  e2e_acc
          single: XGB      0.5966   0.6125  0.6580  0.6002   0.9068
  dyn OLA-cw k=30 p=8      0.5975   0.6118  0.6609  0.5932   0.9049
         dyn LCA k=30      0.5890   0.6066  0.6804  0.5782   0.9060
      dyn OLA-cw k=10      0.5891   0.6060  0.6605  0.5847   0.9036
  dyn OLA-cw k=30 p=3      0.5894   0.6060  0.6579  0.5856   0.9039
         dyn OLA k=10      0.5891   0.6057  0.6600  0.5846   0.9036
 dyn OLA-cw+dist k=30      0.5895   0.6057  0.6579  0.5851   0.9035
         dyn OLA k=20      0.5884   0.6054  0.6603  0.5840   0.9033
          dyn OLA k=5      0.5894   0.6052  0.6557  0.5853   0.9039
         dyn LCA k=10      0.5864   0.6048  0.6723  0.5771   0.9060
      dyn OLA-cw k=30      0.5883   0.6047  0.6564  0.5844   0.9034
         dyn OLA k=50      0.5857   0.6025  0.6535  0.5830   0.9032
static: equal average      0.5851   0.6019  0.6512  0.5828   0.9030
           single: RF      0.5580   0.5780  0.63

## 9. Findings

**Negative result: the dynamically weighted ensemble does not beat the single α=0.25 XGBoost, and
should not be adopted.**

| Configuration | SEL | **Test** |
|---|---|---|
| Single XGB (train-only) — the like-for-like baseline | 0.6167 | **0.6125** |
| Best dynamic (OLA-cw k=30, p=8) | **0.6212** | 0.6118 |
| Static equal-weight average | 0.6090 | 0.6019 |
| Single RF | 0.5984 | 0.5780 |
| Single ET | 0.5889 | 0.5597 |

The dynamic combiner led on SEL by +0.0045 over the best single model, and that lead **did not
transfer**: on test it comes in at −0.0007, ranked 2nd of 15. The SEL advantage was selection noise.
This is precisely what the SEL/test separation was built to catch — under the old protocol, reading
the best row off the test table would have reported "dynamic ensembling improves macro-F1", and it
does not.

### 1. The RF/XGBoost complementarity was an artifact of `balanced` weighting

This is the most consequential finding here, and it **corrects an earlier project claim**. The
motivation for ensembling was that RF and XGBoost had *inverted* precision/recall profiles. Under
α=0.25 that inversion is **gone** — all three families now have the same shape:

| Model (SEL, Stage B) | macro-P | macro-R | macro-F1 |
|---|---|---|---|
| XGB | 0.6641 | 0.5665 | 0.5995 |
| RF | 0.6740 | 0.5273 | 0.5782 |
| ET | 0.6412 | 0.5232 | 0.5665 |

Precision exceeds recall for every model. The earlier "RF = precision / XGB = recall" split was a
property of the **operating point**, not of the algorithms: `balanced` weighting pushed XGBoost into
a high-recall regime and the two models appeared complementary. Fix the weighting and they simply
line up, with XGBoost uniformly ahead.

The disagreement analysis says the same thing. XGB and RF differ on only 13.2% of attack rows, and
when they do, **XGB is right 44.8% of the time against RF's 29.6%** — a ~1.5:1 dominance, not a
complementary trade. There is no "combine RF's precision with XGBoost's recall" to be had, because
that description of the two models is no longer true.

### 2. Large oracle headroom, none of it reachable

The oracle — correct whenever *any* base model is correct — scores **0.6799** against 0.5995 for the
best single model, i.e. **+0.0804 of theoretical headroom**, with at least one model right on 79.1%
of attack rows. So the disagreements do carry usable information in principle.

No competence estimator got at it. The reason is visible in §7.1: the per-instance weights barely
move.

| Model | mean weight | std across instances | min | max | % of rows it leads |
|---|---|---|---|---|---|
| XGB | 0.3348 | 0.0235 | 0.2194 | 0.5565 | 43.4% |
| RF | 0.3321 | 0.0129 | 0.1871 | 0.4123 | 12.6% |
| ET | 0.3331 | 0.0163 | 0.2024 | 0.4177 | 44.0% |

Weights sit at 0.333 ± 0.02 — **a static ensemble in disguise**, which is why every dynamic variant
lands within noise of the equal-weight average. Worse, where the estimator *does* discriminate it
discriminates badly: **ET leads on 44% of rows despite being the weakest model overall**, while RF
leads on only 12.6%. Local accuracy over k=30 neighbours drawn from 6,494 DSEL rows is too noisy an
estimate of competence to beat the prior "just use XGBoost."

That gap — 8 points of oracle headroom, 0 points realised — is the standard oracle-vs-achievable
result, and it is the honest answer to "can dynamic selection help here?": **not with local-accuracy
competence on this data.**

### 3. Static averaging is actively harmful

Equal-weight fusion scores 0.6019 on test against 0.6125 for XGBoost alone. Averaging in two weaker,
non-complementary models dilutes the strong one. Combined with finding 1, this closes the ensemble
direction for Stage B: there is no diversity to exploit, so every fusion rule from plain averaging to
per-instance weighting either does nothing or does harm.

### 4. Per-class effect confirms it

Against the α=0.25 single model, the dynamic ensemble is worse on Shellcode (0.314 vs 0.363), DoS
(0.404 vs 0.411), Reconnaissance (0.617 vs 0.641), Exploits (0.772 vs 0.782) and Fuzzers (0.702 vs
0.713); it is better only on Analysis (0.472 vs 0.439) and Backdoor (0.640 vs 0.599) — the two
smallest families, where a handful of rows moves the score. Not a pattern worth building on.

### Recommendation

**Keep the single α=0.25 XGBoost Stage B (test macro-F1 0.6157).** Do not adopt the ensemble: it
costs a kNN query per flow at inference plus DSEL resident in memory, and returns nothing.

For the thesis this is a reportable negative result with a mechanism, not a dead end — and it is the
second one in a row pointing the same direction. Focal weighting failed because the hard samples are
label-ambiguous; dynamic ensembling fails because the base models are not complementary once the
weighting is right. Both say the remaining error is **shared** across models and hyper-parameters —
i.e. a property of the data rather than of the classifier. That makes the **label-ambiguity ceiling
analysis** the natural next step: it would measure directly what these two experiments have now twice
implied.

**Next levers, unchanged in priority:** per-class decision thresholds (still untried, still the
largest expected gain), gate threshold, soft routing, then the ambiguity ceiling.
